# anatomix + ConvexAdam (legacy ICLR'25 registration)

This tutorial uses the **retained ConvexAdam backend** that produced the anatomix ICLR'25 registration results. It is kept in the repository under `registration_backend/convexadam/` and is **not** the current default backend — for new work use the FireANTs tutorial. Runs on a Colab **T4**.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Install anatomix

ConvexAdam needs only anatomix (no FireANTs backend).

In [ ]:
# Run top-to-bottom once. Clones anatomix and installs it WITHOUT reinstalling
# Colab's torch (so the runtime does not need a restart).
import os
if not os.path.isdir("/content/anatomix"):
    !git clone https://github.com/neel-dey/anatomix.git /content/anatomix
%cd /content/anatomix
!pip -q install -e . --no-deps
!pip -q install monai "dynamic-network-architectures==0.4.4" nibabel SimpleITK \
    torchio natsort huggingface_hub scikit-learn scipy scikit-image matplotlib

## 2. Create the same small synthetic demo pair

In [ ]:
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter, binary_dilation, map_coordinates

def make_demo_data(outdir="demo", size=128, seed=0):
    """Write a small synthetic fixed/moving pair with masks and segmentations.

    The moving image is a smoothly-deformed, contrast-inverted version of the
    fixed image (a cross-modality proxy), so anatomix's contrast-invariant
    features have something meaningful to align.
    """
    os.makedirs(outdir, exist_ok=True)
    rng = np.random.default_rng(seed)
    lin = np.linspace(-1, 1, size)
    zz, yy, xx = np.meshgrid(lin, lin, lin, indexing="ij")
    r = np.sqrt(xx ** 2 + yy ** 2 + zz ** 2)

    outer = np.sqrt((xx / 0.7) ** 2 + (yy / 0.6) ** 2 + (zz / 0.65) ** 2) < 1
    inner = r < 0.28
    seg = outer.astype(np.int16) + inner.astype(np.int16)  # 0 bg, 1 shell, 2 core

    texture = gaussian_filter(rng.standard_normal((size,) * 3), 3)
    fixed = outer * (0.6 + 0.4 * np.sin(6 * xx) * np.cos(6 * yy)) + 0.3 * texture * outer
    fixed = np.clip(fixed, 0, None).astype(np.float32)

    grid = np.stack(np.meshgrid(*[np.arange(size)] * 3, indexing="ij")).astype(np.float32)
    disp = np.stack([gaussian_filter(rng.standard_normal((size,) * 3), 7) * 7
                     for _ in range(3)])
    warped = grid + disp
    moving_base = map_coordinates(fixed, warped, order=1, mode="nearest")
    moving_seg = map_coordinates(seg, warped, order=0, mode="nearest").astype(np.int16)
    fg = moving_seg > 0
    lo = moving_base[fg].min() if fg.any() else 0.0
    hi = moving_base[fg].max() if fg.any() else 1.0
    moving = np.zeros_like(moving_base)
    moving[fg] = (hi - moving_base[fg]) + lo            # contrast inversion in FG

    fixed_mask = binary_dilation(seg > 0, iterations=3).astype(np.uint8)
    moving_mask = binary_dilation(fg, iterations=3).astype(np.uint8)

    aff = np.eye(4)
    for name, arr in [("fixed", fixed), ("moving", moving.astype(np.float32)),
                      ("fixed_mask", fixed_mask), ("moving_mask", moving_mask),
                      ("fixed_seg", seg), ("moving_seg", moving_seg)]:
        nib.save(nib.Nifti1Image(arr, aff), os.path.join(outdir, name + ".nii.gz"))
    print("wrote demo volumes to", outdir, "(size %d^3)" % size)

make_demo_data()

## 3. Register with the retained ConvexAdam backend

`convex_adam` extracts anatomix + MIND-SSC features and runs the coupled-convex + Adam instance-optimization solver. Masks and label warping use the same interface as the paper.

In [ ]:
import os
os.makedirs('ca_out', exist_ok=True)
from anatomix.registration.registration_backend.convexadam import convex_adam

convex_adam(
    expname='demo', lambda_weight=0.75, grid_sp=2, disp_hw=1,
    selected_niter=80, selected_smooth=0,
    hf_variant='anatomix', result_path='ca_out',
    fixed_image='demo/fixed.nii.gz', moving_image='demo/moving.nii.gz',
    use_mask=True, fixed_mask='demo/fixed_mask.nii.gz',
    moving_mask='demo/moving_mask.nii.gz',
    warp_seg=True, fixed_seg='demo/fixed_seg.nii.gz',
    moving_seg='demo/moving_seg.nii.gz',
)  # prints Dice

## 4. Inspect the results

ConvexAdam writes a moved volume, a displacement field, and a warped label map into `ca_out/` (and prints Dice above).

In [ ]:
import glob
import nibabel as nib
import matplotlib.pyplot as plt
fixed = nib.load("demo/fixed.nii.gz").get_fdata()
moving = nib.load("demo/moving.nii.gz").get_fdata()
moved = nib.load(sorted(glob.glob("ca_out/moved_*.nii.gz"))[0]).get_fdata()
z = fixed.shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (im, t) in zip(ax, [(fixed, "fixed"), (moving, "moving"), (moved, "moved")]):
    a.imshow(im[:, :, z].T, cmap="gray", origin="lower"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

To reproduce the ICLR'25 numbers on real data, use ROI masks and CT clipping (`fixed_minclip=-450`, `fixed_maxclip=450`) and register MR to CT, exactly as in the paper. This backend is retained for that purpose; the FireANTs CLI is the general-purpose successor.